In [1]:
import re
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

In [2]:
plp = pd.read_parquet("../data/datasets/PLP_68_2024_textos_emendas.parquet")
plp = plp[plp.num_emenda <= 1998]

In [3]:


def extract_until_any(text, stop_words):
    lower_text = text.lower()
    matches = []

    for word in stop_words:
        idx = lower_text.find(word.lower())
        if idx != -1:
            matches.append(idx)

    if not matches:
        print('---->', text)
        return text

    first_stop = min(matches)
    return text[:first_stop]


def remove_random_chars(text):
    pattern = r"[`´\^¨~]"
    return re.sub(pattern, "", text)

def remove_title(text):
    pattern = r'^([\s\r\n]*\d+[\s\r\n]+)?([\s\r\n]*Senado Federal[\s\r\n]*)?(?:(?:Gabinete)?\s*(?:d[oa]\s*)?Senadora?\s+[^\r\n]+[\s\r\n]+)?([\s\r\n]+\d+[\s\r\n]+)?(?:Emenda\s+da\s+CCJ[\s\r\n]+)?EMENDA\s+(?:N|n)[º°o]?\s*[,-–—_\s]*\s*(?:CCJ|)?\s*\r?\n?\s*(\(?ao\s+(PLP|Projeto[\s\r\n]+de[\s\r\n]+lei[\s\r\n]+complementar)\s*(?:n[º°o]\s*)?68(?:\/|,\s*de\s*)2024\.?\)?)?\s*\r?\n?'
    return re.sub(pattern, '=====================', text, flags=re.IGNORECASE | re.MULTILINE)


def remove_title(text):
    pattern = r'^([\s\S]*?)\((ao)?\s*(?:PLP|Projeto de lei complementar)\s*(?:n[º°o]\s*)?68(?:\/|,\s*de\s*)2024\)'
    return re.sub(pattern, '=====================', text, flags=re.IGNORECASE | re.MULTILINE)

def remove_address(text):
    pattern = r'(?:Praça\s+dos\s+Três\s+Poderes[\s\S]{0,200}?Senado\s+Federal[\s\S]{0,200}?(?:Gabinete\s*\d+|Gabinete\s+\d+|Gabinete\s*[A-Z0-9/.-]+))|(?:SENADO\s+FEDERAL|Senado\s+Federal)[\s|–-]+[\s\S]*?(?:(?:CEP\s*)?\d{2}\.?\d{3}-\d{3}[\s|–/-]*Brasília(?:\s*[–/-]?\s*DF)?|Brasília(?:\s*[–/-]?\s*DF)?[\s|–/-]*(?:CEP\s*)?\d{2}\.?\d{3}-\d{3})(?:[\s\S]{0,300}?(?:Telefone|Fone|Fax|E-?mail)\s*:?[^\n|]*)*(?:[\s\S]{0,150}?[A-Za-z0-9._%+-]+@senad(?:o|or)\.leg\.br)?\.?|(?:Gabinete\s+do\s+Senador[a]?\s+[^\n-]+[\s–|-]*?(?:Telefone|Fone)\s*:\s*\+?\d[\d\s().-]+)\.?'
    return re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)

def remove_gabinete(text):
    pattern = r'([\s\r\n]*Senado Federal[\s\r\n]*)?(?:Gabinete\s*(?:d[oa]\s*)?Senadora?\s+[a-zA-Zà-úÀ-Ú ]+[\s\r\n]+)'
    return re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)

def extract_after_pl(text):
    pattern = r'(?:\((?:ao)?\s*(?:substitutivo ao)?\s*(?:Projeto de Lei|PLP?|Projeto de Lei Complementar)\s*(?:n[ºo]\.?)?\s*[\d,\.]+/?(?:,?\s*de\s*|\s*/)\d{4}\))(.*)'
    match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return text  # Return original text if no match

def remove_lexedit_id(text):
    cleaned = re.sub(r'SF\/\d+\.\d+-\d{2}\s*(\(LexEdit\*?\))?', '', text)
    return cleaned

def remove_digital_signature(text):
    pattern = r'Assinado eletronicamente, por .*?\nPara verificar as assinaturas, acesse https://legis\.senado\.gov\.br/autenticadoc-legis/\d+'
    cleaned = re.sub(pattern, '', text)
    return cleaned.strip()

def remove_num_emenda(text):
    pattern = r'^\s*0\d+(?:-U)?\s*PLP 68\/2024\n?'
    
    return re.sub(pattern, '', text, flags=re.MULTILINE | re.IGNORECASE).strip()


## Pré-processamento

### Remover cabeçalho

#### Padrão geral

In [4]:
plp[plp.texto.str.contains("(ao PLP 68/2024)")]

C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_21364\3349167471.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  plp[plp.texto.str.contains("(ao PLP 68/2024)")]


,num_emenda,materia,nome_arquivo,texto
0,1,PLP_68_2024,EMENDA_1-U_-_PLP_68_2024.txt,Gabinete do Senador Esperidião Amin\nEMENDA Nº...
1,10,PLP_68_2024,EMENDA_10-U_-_PLP_68_2024.txt,Gabinete do Senador Mecias de Jesus\nEMENDA Nº...
2,100,PLP_68_2024,EMENDA_100-U_-_PLP_68_2024.txt,Gabinete do Senador Fabiano Contarato\nEMENDA ...
3,1000,PLP_68_2024,EMENDA_1000-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nAcrescen...
4,1001,PLP_68_2024,EMENDA_1001-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nDê-se ao...
...,...,...,...,...
2234,995,PLP_68_2024,EMENDA_995-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nDê-se ao...
2235,996,PLP_68_2024,EMENDA_996-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nAcrescen...
2236,997,PLP_68_2024,EMENDA_997-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nAcrescen...
2237,998,PLP_68_2024,EMENDA_998-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nDê-se no...


In [5]:
default_header_pattern = r'^([\s\S]*?\(ao\s*(?:PLP|Projeto de lei complementar)\s*(?:n[º°o]\s*)?68(?:\/|,\s*de\s*)2024\)(?:\s*\(ao\s*Projeto\s+de\s+Lei\s+Complementar\s*(?:n[º°o]\s*)?68,\s*de\s*2024\))?)'

plp["cabecalho"] = plp["texto"].str.extract(
    default_header_pattern,
    flags=re.IGNORECASE
)[0]

In [6]:
plp["texto_sem_cabecalho"] = plp["texto"].str.replace(
    default_header_pattern,
    "",
    regex=True,
    flags=re.IGNORECASE
)

In [7]:
plp[plp.cabecalho.str.len() < 100].cabecalho.values

array(['Gabinete do Senador Esperidião Amin\nEMENDA Nº         \n(ao PLP 68/2024)',
       'Gabinete do Senador Mecias de Jesus\nEMENDA Nº         \n(ao PLP 68/2024)',
       'Gabinete do Senador Fabiano Contarato\nEMENDA Nº         \n(ao PLP 68/2024)',
       ..., 'EMENDA Nº         \n(ao PLP 68/2024)',
       'EMENDA Nº         \n(ao PLP 68/2024)',
       'EMENDA Nº         - CCJ\n(ao PLP 68/2024)'],
      shape=(1871,), dtype=object)

#### Exceções

In [8]:
len(plp[plp.cabecalho.isna()])

27

In [9]:
for t in plp[(plp.cabecalho.isna())].texto.values:
    print(t[:500])
    print("+=========================================================")

 
 
 
 
 
Gabinete do Senador Jaime Bagattoli 
 
 
 
 
 
 
Senado Federal –Anexo 2, Ala Teotônio Vilela, Gabinete 23 - Praça dos Três Poderes – CEP 70165-900 – Brasília DF  
Gabinete do Senador Jaime Bagattoli -Telefone: +55 (61) 3303-2714 
 
EMENDA nº  
 
O artigo 9º, inciso III, alínea “b” do projeto passa a ter a redação seguinte: 
 
Art.  9º - ................................................................................ 
III ................................................................
+=========================================================
 
 
 
 
Senado Federal – Anexo I – 3º Andar – Praça dos Três Poderes – CEP 70165-900 – Brasília DF  
Telefone: +55 (61) 3303-6190 – sen.cironogueira@senado.leg.br 
EMENDA Nº       AO PROJETO DE LEI 
COMPLEMENTAR Nº 68/2024. 
 
Acrescente-se inciso III ao caput do art. 136 do Projeto, com a 
seguinte redação:  
“Art.136. ................................................................................... 
.................................

In [10]:
set(plp["texto_sem_cabecalho"].str.split().str[0].values)

{'(Do',
 '1',
 '1)',
 'A',
 'ANEXO',
 'Acrescenta',
 'Acrescenta-se',
 'Acrescentam-se',
 'Acrescente',
 'Acrescente-se',
 'Acrescente-se,',
 'Acrescentem-se',
 'Acrescentem-se,',
 'Acresça-se',
 'Acresçam-se',
 'Ajuste-se',
 'Altera',
 'Altera-se',
 'Alteram-se',
 'Alteração',
 'Altere-se',
 'Altere-se,',
 'Alterem-se',
 'Anexo',
 'Art.',
 'Art.1º',
 'Art.283.................................................................................................................................',
 'Atribua-se',
 'Confira-se',
 'Determine-se',
 'Dá',
 'Dá-se',
 'Dê',
 'Dê-se',
 'Dêem-se',
 'EMENDA',
 'Exclua-se',
 'Excluam-se',
 'Fica',
 'Ficam',
 'Gabinete',
 'Inclua-se',
 'Inclua-se,',
 'Incluam-se',
 'Inclui',
 'Inclui-se',
 'Inclusão',
 'Insere-se',
 'Inserir',
 'Insira-se',
 'Insiram-se',
 'Institui',
 'Item',
 'Modifica',
 'Modifica-se',
 'Modificação',
 'Modifique-se',
 'No',
 'O',
 'Os',
 'PROJETO',
 'SENADO',
 'Senado',
 'Substitua-se',
 'Suprima-se',
 'Suprimam-se',
 'Suprimam–se',
 '

In [11]:
import re

CORRECT_START_PATTERNS = [
    r"O artigo\b",
    r"O art\.",
    r"\bOs arts?\.",
    r"\bO inciso\b",
    r"\bAcrescente-se\b",
    r"\bAcrescentem-se\b",
    r"\bAcrescente\b",
    r"\bAcrescenta-se\b",
    r"\bAcrescenta\b",
    r"\bAcrescentam-se\b",
    r"\bAcresça-se\b",
    r"\bAcresçam-se\b",
    r"\bInclua-se\b",
    r"\bIncluam-se\b",
    r"\bInclui\b",
    r"\bInclui-se\b",
    r"\bInclusão\b",
    r"\bInserir\b",
    r"\bInsira-se\b",
    r"\bInsiram-se\b",
    r"\bInsere-se\b",
    r"\bSuprima-se\b",
    r"\bSuprimam-se\b",
    r"\bSubstitua-se\b",
    r"\bDê-se\b",
    r"\bDê\b",
    r"\bDêem-se\b",
    r"\bDá-se\b",
    r"\bAltera\b",
    r"\bAltera-se\b",
    r"\bAlterem-se\b",
    r"\bAlteram-se\b",
    r"\bAltere-se\b",
    r"\bModifique-se\b",
    r"\bModifica\b",
    r"\bModifica-se\b",
    r"\bAjuste-se\b",
    r"\bDetermine-se\b",
    r"\bConfira-se\b",
    r"\bExclua-se\b",
    r"\bExcluam-se\b",
    r"\bAtribua-se\b",
    r"\bFica\b",
    r"\bFicam\b",
    r"\bSugerimos\b",
]

except_header_pattern = re.compile(
    rf"(?:{'|'.join(CORRECT_START_PATTERNS)})",
    re.IGNORECASE | re.MULTILINE
)

In [12]:
def get_exception_header(text, pattern=except_header_pattern):
    if not isinstance(text, str):
        return None, text

    m = pattern.search(text)

    if not m:
        return None, text

    idx = m.start()

    return text[:idx].strip(), text[idx:].strip()

In [13]:
mask = plp["cabecalho"].isna()

resultado = plp.loc[mask, "texto"].apply(get_exception_header)

plp.loc[mask, "cabecalho"] = resultado.str[0].to_numpy()
plp.loc[mask, "texto_sem_cabecalho"] = resultado.str[1].to_numpy()

In [14]:
plp[plp["cabecalho"].isna()]

,num_emenda,materia,nome_arquivo,texto,cabecalho,texto_sem_cabecalho


#### Remover ementa

In [15]:
print(len(plp[plp["texto_sem_cabecalho"].str.contains("Institui o Imposto sobre Bens")]))

55


In [16]:
plp[plp["texto_sem_cabecalho"].str.contains("Institui o Imposto sobre Bens")].texto_sem_cabecalho.str[:500].values

array([' \nInstitui o Imposto sobre Bens e Serviços - \nIBS, a Contribuição Social sobre Bens e \nServiços - CBS e o Imposto Seletivo - IS e dá \noutras providências. \n \nArt. 1º. Suprima-se o artigo 35 do Projeto de Lei Complementar nº \n68 de 2024.  \nJUSTIFICAÇÃO \nA ausência de prazo para a utilização dos créditos tributários visa \nproporcionar maior flexibilidade e segurança jurídica aos contribuintes.  \nEm muitos casos, o prazo limitado para a utilização de créditos pode \nimpedir que empresas com fluxos d',
       ' \nInstitui o Imposto sobre Bens e Serviços - \nIBS, a Contribuição Social sobre Bens e \nServiços - CBS e o Imposto Seletivo - IS e dá \noutras providências. \n \nArt. 1º. Adicione-se o seguinte dispositivo do Projeto de Lei \nComplementar nº 68 de 2024: \n \nArt. 58. ............................................................................................... \n................................................................................................ \n§1

In [17]:
def remover_ementa(texto):
    pattern = (
    r'^\s*institui\s+o\s+imposto\s+sobre\s+bens\s+e\s+serviços[\s\S]*?provid[eê]ncias\.\s*'
    )
    
    return re.sub(
        pattern,
        '',
        texto,
        flags=re.IGNORECASE
    )

In [18]:
plp[plp["texto_sem_cabecalho"].str.contains("Institui o Imposto sobre Bens")].texto_sem_cabecalho.apply(remover_ementa).str[:50].values

array(['Art. 1º. Suprima-se o artigo 35 do Projeto de Lei ',
       'Art. 1º. Adicione-se o seguinte dispositivo do Pro',
       'Art. 1º. Adicione-se o seguinte dispositivo do Pro',
       'Art. 1º. Adicione-se o seguinte dispositivo do Pro',
       '\nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMENDA Nº ',
       '\nAcrescente-se um novo Artigo nº 509-A, ao Projeto',
       '\nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMENDA Nº ',
       '\nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMENDA Nº ',
       '\nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMENDA Nº ',
       '\nDê-se ao art. 125 da Proposta de Lei Complementar',
       '\nAcrescente-se art. 122-1 ao Capítulo II do Título',
       '\nEMENDA Nº   - CCJ\n(ao PLP 68/2024)\nPROPOSTA DE EM',
       'Sugerimos a inclusão dos artigos 37-A e 37-B na Se',
       'Sugerimos nova redação, com alteração da redação d',
       'Inclua-se o seguinte artigo ao PLP 68/2024: \n \nSeç',
       'Dê-se nova redação ao seguinte dispositivo: \n \nArt',
      

In [19]:
plp["texto_sem_cabecalho"] = plp.texto_sem_cabecalho.apply(remover_ementa)

#### Checagem da primeira palavra do texto da emenda

In [20]:
set(plp["texto_sem_cabecalho"].str.split().str[0].values)

{'(Do',
 '1)',
 'A',
 'ANEXO',
 'Acrescenta',
 'Acrescenta-se',
 'Acrescentam-se',
 'Acrescente',
 'Acrescente-se',
 'Acrescente-se,',
 'Acrescentem-se',
 'Acrescentem-se,',
 'Acresça-se',
 'Acresçam-se',
 'Ajuste-se',
 'Altera',
 'Altera-se',
 'Alteram-se',
 'Alteração',
 'Altere-se',
 'Altere-se,',
 'Alterem-se',
 'Anexo',
 'Art.',
 'Art.1º',
 'Art.283.................................................................................................................................',
 'Atribua-se',
 'Confira-se',
 'Determine-se',
 'Dá',
 'Dá-se',
 'Dê',
 'Dê-se',
 'Dêem-se',
 'EMENDA',
 'Exclua-se',
 'Excluam-se',
 'Fica',
 'Ficam',
 'Inclua-se',
 'Inclua-se,',
 'Incluam-se',
 'Inclui',
 'Inclui-se',
 'Inclusão',
 'Insere',
 'Insere-se',
 'Inserir',
 'Insira-se',
 'Insiram-se',
 'Item',
 'Modifica',
 'Modifica-se',
 'Modificação',
 'Modifique-se',
 'No',
 'O',
 'Os',
 'PROJETO',
 'Substitua-se',
 'Sugerimos',
 'Suprima-se',
 'Suprimam-se',
 'Suprimam–se',
 'São',
 'a)',
 '“Art.'}

In [21]:
palavras_suspeitas = ['(Do',
 #'1)',
 #'A',
 #'ANEXO',
 #'Anexo',
 'EMENDA',
 #'Item',
 #'No',
 #'O',
 #'Os',
'PROJETO',
 #'São',
 #'a)',
]

for palavra in palavras_suspeitas:
    s = plp[plp["texto_sem_cabecalho"].str.split().str[0] == palavra].texto.values
    print(f"{'-'*50}Palavra: {palavra}{'-'*50}")
    s = set([t[:400] for t in s])
    for texto in s:
        print(texto)
        print("===============================================")
    print("---------------------------------------------------------------------------------------------------")

--------------------------------------------------Palavra: (Do--------------------------------------------------
Gabinete do Senador Zequinha Marinho
EMENDA Nº         
(ao PLP 68/2024)
(Do Sr. Senador Zequinha Marinho) 
   Inclua-se os seguintes itens no Anexo IX do Projeto de Lei
Complementar nº 68 de 2024:
      
7
Inseticidas, fungicidas,
formicidas, herbicidas,
parasiticidas, germicidas,
acaricidas, nematicidas,
raticidas, desfolhantes,
dessecantes, espalhantes,
adesivos, estimuladores e
inibidores de c
---------------------------------------------------------------------------------------------------
--------------------------------------------------Palavra: EMENDA--------------------------------------------------
 
 
 
 
SENADO FEDERAL  
Gabinete do Senador Mecias de Jesus  
 
 
 
Praça dos Três Poderes – Senado Federal – Anexo II – Ala Ruy Carneiro – Gabinete 02 
 
 
EMENDA Nº             , DE 2024  
(ao Projeto de Lei Complementar nº 68, de 2024) 
 
EMENDA MODIFICATIVA 
Inclua

##### Remover restante dos dados de cabeçalho

In [22]:
def remover_dados_restantes_cabecalho(texto):
    extrair_ate = [
        r"EMENDA SUPRESSIVA\s*n[º°o]?\s*,?\s*DE\s*2024\.?",
        r"EMENDA MODIFICATIVA Nº",
        r"EMENDA MODIFICATIVA",
        r"EMENDA SUBSTITUTIVA",
        r"EMENDA DE REDAÇÃO",
        r"\(Do Sr\. Senador Zequinha Marinho\)",
        r"\n\s*EMENDA\s*\n"
    ]
    
    pattern = re.compile(
        rf"^[\s\S]*?(?:{'|'.join(extrair_ate)})\s*\n*",
        flags=re.IGNORECASE
    )
    
    return pattern.sub('', texto, count=1)

In [23]:
plp[plp["texto_sem_cabecalho"].str.split().str[0].isin(palavras_suspeitas)]["texto_sem_cabecalho"]

196     \nEMENDA DE REDAÇÃO\nDê-se ao item 24 do Anexo...
201     \nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMEN...
212     \nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMEN...
224     \nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMEN...
235     \nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMEN...
538     \nEMENDA SUPRESSIVA nº         , DE 2024.\nSup...
1446    \nEMENDA MODIFICATIVA\n \nArt. 1º. Suprima-se ...
1522    \n(Do Sr. Senador Zequinha Marinho) \n   Inclu...
1580    \nEMENDA Nº   - CCJ\n(ao PLP 68/2024)\nPROPOST...
1653    \nEMENDA SUBSTITUTIVA\nCAPÍTULO I\nDAS DISPOSI...
1912     \n \nEMENDA MODIFICATIVA \n \nMigra os §§4º e...
1913     \n \nEMENDA MODIFICATIVA \n \n \nO art. 419, ...
2068    \nPROJETO DE LEI COMPLEMENTAR Nº 68/2024\nEMEN...
2221     \n \nEMENDA MODIFICATIVA \nInclua-se o §6º no...
2223     \n \n \n \n \nEMENDA MODIFICATIVA \n \n \nO A...
Name: texto_sem_cabecalho, dtype: object

In [24]:
mask = plp["texto_sem_cabecalho"].str.split().str[0].isin(palavras_suspeitas)

plp.loc[mask, "texto_sem_cabecalho"] = (
    plp.loc[mask, "texto_sem_cabecalho"]
       .apply(remover_dados_restantes_cabecalho)
)

In [25]:
plp.loc[mask].texto_sem_cabecalho

196     Dê-se ao item 24 do Anexo III - Serviços de Sa...
201     Institui o Imposto sobre Bens e Serviços IBS, ...
212     Institui o Imposto sobre Bens e Serviços IBS, ...
224     Institui o Imposto sobre Bens e Serviços IBS, ...
235     Institui o Imposto sobre Bens e Serviços IBS, ...
538     Suprima-se a expressão "médicos veterinários" ...
1446    Art. 1º. Suprima-se o Parágrafo Único do art. ...
1522    Inclua-se os seguintes itens no Anexo IX do Pr...
1580    Dê-se à alínea “b” do item I do artigo 411; ao...
1653    CAPÍTULO I\nDAS DISPOSIÇÕES PRELIMINARES\nArt....
1912    Migra os §§4º e 5º do artigo 420 para o artigo...
1913    O art. 419, §1º, inciso II passa a constar com...
2068    Art. 1º. Inclua-se e ajuste-se os seguintes it...
2221    Inclua-se o §6º no artigo 23 e altere-se incis...
2223    O Art. 450 do Projeto de Lei Complementar nº 6...
Name: texto_sem_cabecalho, dtype: object

In [26]:
set(plp["texto_sem_cabecalho"].str.split().str[0].values)

{'1)',
 'A',
 'ANEXO',
 'Acrescenta',
 'Acrescenta-se',
 'Acrescentam-se',
 'Acrescente',
 'Acrescente-se',
 'Acrescente-se,',
 'Acrescentem-se',
 'Acrescentem-se,',
 'Acresça-se',
 'Acresçam-se',
 'Ajuste-se',
 'Altera',
 'Altera-se',
 'Alteram-se',
 'Alteração',
 'Altere-se',
 'Altere-se,',
 'Alterem-se',
 'Anexo',
 'Art.',
 'Art.1º',
 'Art.283.................................................................................................................................',
 'Atribua-se',
 'CAPÍTULO',
 'Confira-se',
 'Determine-se',
 'Dá',
 'Dá-se',
 'Dê',
 'Dê-se',
 'Dêem-se',
 'Exclua-se',
 'Excluam-se',
 'Fica',
 'Ficam',
 'Inclua-se',
 'Inclua-se,',
 'Incluam-se',
 'Inclui',
 'Inclui-se',
 'Inclusão',
 'Insere',
 'Insere-se',
 'Inserir',
 'Insira-se',
 'Insiram-se',
 'Institui',
 'Item',
 'Migra',
 'Modifica',
 'Modifica-se',
 'Modificação',
 'Modifique-se',
 'No',
 'O',
 'Os',
 'Substitua-se',
 'Sugerimos',
 'Suprima-se',
 'Suprimam-se',
 'Suprimam–se',
 'São',
 'a)',
 '“Art.'}

### Remover dados do corpo

#### Remover gabinete

In [27]:
print(len(plp[plp.texto_sem_cabecalho.str.contains(r"Gabinete (?:d[oa]\s)?Sen", case=False)]))

100


In [28]:
trechos_com_gabinete = (
    plp.loc[
        plp["texto_sem_cabecalho"].str.contains("Gabinete (?:d[oa]\s)?Sen", na=False, case=False),
        "texto_sem_cabecalho"
    ]
    .str.extract(r"(.{0,20}Gabinete (?:d[oa]\s)?Sen.{0,20})", expand=False)
)

print(trechos_com_gabinete.tolist())

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_21364\3161471891.py:3: SyntaxWarning: invalid escape sequence '\s'
  plp["texto_sem_cabecalho"].str.contains("Gabinete (?:d[oa]\s)?Sen", na=False, case=False),


['Gabinete Senador Carlos Portinho', 'Gabinete do Senador NELSINHO TRAD ', 'Gabinete do Senador NELSINHO TRAD ', 'Gabinete do Senador NELSINHO TRAD ', 'Gabinete do Senador NELSINHO TRAD ', 'Gabinete do Senador Jaime Bagattoli', 'Gabinete do Senador Jaime Bagattoli', 'Gabinete do Senador ESPERIDIÃO AMIN', 'Gabinete do Senador ESPERIDIÃO AMIN', 'Gabinete do Senador Jaime Bagattoli', 'Gabinete do Senador Jaime Bagattoli', 'Gabinete do Senador Jaime Bagattoli', 'Gabinete do Senador Ciro Nogueira (', 'Gabinete do Senador FLÁVIO ARNS ', 'Gabinete do Senador FLÁVIO ARNS ', 'Gabinete do Senador FLÁVIO ARNS ', 'Gabinete do Senador Fabiano Contara', 'Gabinete do Senador Fabiano Contara', 'Gabinete do Senador Fabiano Contara', 'Gabinete do Senador Beto Martins ', 'Gabinete do Senador Beto Martins ', 'Gabinete do Senador Beto Martins ', 'Gabinete do Senador Mecias de Jesus', 'Gabinete do Senador Mecias de Jesus', 'Gabinete do Senador Mecias de Jesus', 'Gabinete do Senador Mecias de Jesus', 'Gabine

In [29]:
mask = plp["texto_sem_cabecalho"].str.contains(r"Gabinete (?:d[oa]\s)?Sen", na=False, case=False)

# linhas com "Gabinete d"
plp.loc[mask, "texto_preprocessado"] = (
    plp.loc[mask, "texto_sem_cabecalho"]
       .apply(remove_gabinete)
)

# linhas sem "Gabinete d"
plp.loc[~mask, "texto_preprocessado"] = (
    plp.loc[~mask, "texto_sem_cabecalho"]
)

trechos_restantes = (
    plp.loc[
        plp["texto_preprocessado"].str.contains(r"Gabinete (?:d[oa]\s)?Sen", na=False, case=False),
        "texto_preprocessado"
    ]
    .str.findall(r".{0,20}Gabinete (?:d[oa]\s)?Sen.{0,20}")
)

print(trechos_restantes.tolist())

[]


#### Tag do LexEdit

In [30]:
plp.head().texto_preprocessado

0    \nSuprimam-se os §§ 1° e 2° do art. 384 do PLP...
1    \nInclua-se o §6º no art. 23 e dê-se a seguint...
2    \nAcrescenta o inciso VIII ao §1º do artigo 40...
3    \nAcrescente-se inciso VI ao parágrafo único d...
4    \nDê-se ao § 7º do art. 163 do Projeto a segui...
Name: texto_preprocessado, dtype: object

In [31]:
print(len(plp[plp.texto_sem_cabecalho.str.contains("SF/")]))

1998


In [32]:
plp[plp.texto_sem_cabecalho.str.contains("SF/")].texto_sem_cabecalho.values[0]

'\nSuprimam-se os §§ 1° e 2° do art. 384 do PLP nº 68, de 2024,\nrenumerando-se os demais, e inclua-se a alínea “d” ao inciso IV do caput desse\ndispositivo, com a seguinte redação:\n“Art. 384.............................................................................\n............................................................................................\nIV –...................................................................................\n............................................................................................\nd) recolhimento a fundo estadual ou distrital como contrapartida para\nfruição de incentivo ou benefício ﬁscal.\n............................................................................................”\nJUSTIFICAÇÃO\nEsta Emenda visa corrigir a distorção aprovada pela Câmara dos\nDeputados que não considera a contribuição aos fundos estaduais e distrital como\numa condição onerosa, o que implica subtrair do contribuinte o direito de se\nressar

In [33]:
plp["texto_preprocessado"] = plp.texto_preprocessado.apply(remove_lexedit_id)

In [34]:
print(len(plp[plp.texto_preprocessado.str.contains(r"SF/\d")]))

0


#### Assinatura Digital

In [35]:
print(len(plp[plp.texto_preprocessado.str.contains("Assinado eletronicamente")]))

1907


In [36]:
plp["texto_preprocessado"] = plp.texto_preprocessado.apply(remove_digital_signature)

In [37]:
print(len(plp[plp.texto_preprocessado.str.contains("Assinado eletronicamente")]))

0


#### Endereço

In [38]:
plp[plp.texto_preprocessado.str.contains("Praça|CEP|Ala|Gabinete")].texto_preprocessado

2       Acrescenta o inciso VIII ao §1º do artigo 406 ...
27      Art. 1º. Suprima-se o artigo 35 do Projeto de ...
28      Art. 1º. Adicione-se o seguinte dispositivo do...
29      Art. 1º. Adicione-se o seguinte dispositivo do...
30      Art. 1º. Adicione-se o seguinte dispositivo do...
                              ...                        
2225    Dê-se ao artigo 280, do Projeto de Lei Complem...
2226    Dê-se ao artigo 278, do Projeto de Lei Complem...
2227    Inclua-se o art. 248-A no PLP 68, de 2024, com...
2229    Inclua-se o art. 253-A no PLP 68, de 2024, com...
2230    Altera-se a redação do caput do art. 257 do PL...
Name: texto_preprocessado, Length: 108, dtype: object

In [39]:
trechos_com_endereco = (
    plp.loc[
        plp["texto_preprocessado"].str.contains(r"Praça\s|CEP\s|Ala\s|Gabinete\s", na=False),
        "texto_preprocessado"
    ]
    # Regex atualizada que aceita dinamicamente os dois casos
    .str.extractall(r"([\s\S]{0,70}(Praça\s|CEP\s|Ala\s|Gabinete\s)[\s\S]{0,70})")
)

print(trechos_com_endereco[0].tolist())

['alados na referida Região. \n\n01022-U\nPLP 68/2024\n \n \nSenado Federal – Praça dos Três Poderes – Anexo I – 24º andar – CEP 70165-900 – Brasília DF \n', ' das empresas exportadoras \n\n01023-U\nPLP 68/2024\n \n \nSenado Federal – Praça dos Três Poderes – Anexo I – 24º andar – CEP 70165-900 – Brasília DF \n', ' área de livre comércio de \n\n01024-U\nPLP 68/2024\n \n \nSenado Federal – Praça dos Três Poderes – Anexo I – 24º andar – CEP 70165-900 – Brasília DF \n', 'rança jurídica, contábil e \n\n01025-U\nPLP 68/2024\n \n \nSenado Federal – Praça dos Três Poderes – Anexo I – 24º andar – CEP 70165-900 – Brasília DF \n', 'stitucional. \n\n01068-U\nPLP 68/2024\n \n \n \n \n \nSenado Federal –Anexo 2, Ala Teotônio Vilela, Gabinete 23 - Praça dos Três Poderes – CEP 70165-900 ', ' \neconômica. \n\n01069-U\nPLP 68/2024\n \n \n \n \n \nSenado Federal –Anexo 2, Ala Teotônio Vilela, Gabinete 23 - Praça dos Três Poderes – CEP 70165-900 ', 'letivos de trabalho. \n\n01072-U\nPLP 68/2024Senado Fed

In [40]:
plp["texto_preprocessado"] = plp.texto_preprocessado.apply(remove_address)

In [41]:
trechos_com_endereco = (
    plp.loc[
        plp["texto_preprocessado"].str.contains(r"Praça\s|CEP\s|Ala\s|Gabinete\s", na=False),
        "texto_preprocessado"
    ]
    # Regex atualizada que aceita dinamicamente os dois casos
    .str.extractall(r"([\s\S]{0,70}(Praça\s|CEP\s|Ala\s|Gabinete\s)[\s\S]{0,70})")
)

print(trechos_com_endereco[0].tolist())

['ao Ministério do Desenvolvimento, Indústria, Comércio e Serviços\ne ao Gabinete de Segurança Institucional da Presidência da República, revisarão,\na c', 'ao Ministério do Desenvolvimento, Indústria, Comércio e Serviços\ne ao Gabinete de Segurança Institucional da Presidência da República, revisarão,\na c', 'las guardas municipais, pela Agência Brasileira de Inteligência, pelo Gabinete\nde Segurança Institucional da Presidência da República, pelos tribunai']


#### Remove número da emenda

In [42]:
plp["texto_preprocessado"] = plp.texto_preprocessado.apply(remove_num_emenda)

### Remover justificativa

In [43]:
def extract_until_any(texto, stop_words):
    if not texto:
        return texto
    
    # Primeiro passo: padronizar espaçamentos e remover quebras de linhas do PDF
    # Isso une palavras como "JUSTIFICAÇÃ\nO" caso apareçam no resto da base
    texto_normalizado = re.sub(r'(\w+)-\n\s*(\w+)', r'\1\2', texto)
    texto_normalizado = texto_normalizado.replace("JUSTIFICAÇÃ\nO", "JUSTIFICAÇÃO")
    texto_normalizado = texto_normalizado.replace("JUSTIFICAT\nIVA", "JUSTIFICATIVA")
    
    # Criamos um padrão que busca as palavras inteiras (\b) independente de estarem no início da linha
    # O [| ]*? serve para engolir aquela barra vertical " | JUSTIFICAÇÃO" que aparece no texto 5
    padrao = r"[| ]*?\b(?:" + "|".join([re.escape(word) for word in stop_words]) + r")\b"
    
    # Compilamos apenas com IGNORECASE (não precisamos de MULTILINE aqui)
    regex = re.compile(padrao, re.IGNORECASE)
    
    match = regex.search(texto_normalizado)
    
    if match:
        # Corta exatamente antes da palavra de parada (ou antes dos espaços/barras que a antecedem)
        return texto_normalizado[:match.start()].strip()
    
    return texto_normalizado

In [44]:
import re

# 1. Sua lista de variações original
just_variacoes = ["justificação", "justificativa", "j u s t i f i c a ç ã o", "j u s t i f i c a ç ã", "JUSTIFICAÇÃ\nO"]

# 2. Cria a regex dinamicamente colocando ^\s* no início de cada alternativa
# Isso vai gerar: ^\s*justificação|^\s*justificativa|^\s*j\ u\ s...
padrao_dinamico = "|".join([rf"^\s*{re.escape(word)}" for word in just_variacoes])

# 3. Compila com as flags de linha e caixa alta/baixa
regex_validacao = re.compile(padrao_dinamico, re.IGNORECASE | re.MULTILINE)

# 4. Seu loop de teste rodando no que a função antiga 'extract_until_any' gerou:
for idx, texto_filtrado in enumerate(plp.texto_preprocessado.apply(lambda x: extract_until_any(x, just_variacoes))):
    
    # Se a regex achar alguma das palavras iniciando uma linha no texto que deveria estar limpo:
    if regex_validacao.search(texto_filtrado):
        print(f"❌ Erro na PEC {idx}: Uma justificativa ainda foi encontrada no texto cortado!")
    else:
        print(f"✅ PEC {idx}: Passou no teste.")

✅ PEC 0: Passou no teste.
✅ PEC 1: Passou no teste.
✅ PEC 2: Passou no teste.
✅ PEC 3: Passou no teste.
✅ PEC 4: Passou no teste.
✅ PEC 5: Passou no teste.
✅ PEC 6: Passou no teste.
✅ PEC 7: Passou no teste.
✅ PEC 8: Passou no teste.
✅ PEC 9: Passou no teste.
✅ PEC 10: Passou no teste.
✅ PEC 11: Passou no teste.
✅ PEC 12: Passou no teste.
✅ PEC 13: Passou no teste.
✅ PEC 14: Passou no teste.
✅ PEC 15: Passou no teste.
✅ PEC 16: Passou no teste.
✅ PEC 17: Passou no teste.
✅ PEC 18: Passou no teste.
✅ PEC 19: Passou no teste.
✅ PEC 20: Passou no teste.
✅ PEC 21: Passou no teste.
✅ PEC 22: Passou no teste.
✅ PEC 23: Passou no teste.
✅ PEC 24: Passou no teste.
✅ PEC 25: Passou no teste.
✅ PEC 26: Passou no teste.
✅ PEC 27: Passou no teste.
✅ PEC 28: Passou no teste.
✅ PEC 29: Passou no teste.
✅ PEC 30: Passou no teste.
✅ PEC 31: Passou no teste.
✅ PEC 32: Passou no teste.
✅ PEC 33: Passou no teste.
✅ PEC 34: Passou no teste.
✅ PEC 35: Passou no teste.
✅ PEC 36: Passou no teste.
✅ PEC 37: P

In [45]:
def extract_until_any(texto, stop_words):
    if not texto:
        return texto
    
    # Primeiro passo: padronizar espaçamentos e remover quebras de linhas do PDF
    # Isso une palavras como "JUSTIFICAÇÃ\nO" caso apareçam no resto da base
    texto_normalizado = re.sub(r'(\w+)-\n\s*(\w+)', r'\1\2', texto)
    texto_normalizado = texto_normalizado.replace("JUSTIFICAÇÃ\nO", "JUSTIFICAÇÃO")
    texto_normalizado = texto_normalizado.replace("JUSTIFICAT\nIVA", "JUSTIFICATIVA")
    
    # Criamos um padrão que busca as palavras inteiras (\b) independente de estarem no início da linha
    # O [| ]*? serve para engolir aquela barra vertical " | JUSTIFICAÇÃO" que aparece no texto 5
    padrao = r"[| ]*?\b(?:" + "|".join([re.escape(word) for word in stop_words]) + r")\b"
    
    # Compilamos apenas com IGNORECASE (não precisamos de MULTILINE aqui)
    regex = re.compile(padrao, re.IGNORECASE)
    
    match = regex.search(texto_normalizado)
    
    if match:
        # Corta exatamente antes da palavra de parada (ou antes dos espaços/barras que a antecedem)
        return texto_normalizado[:match.start()].strip()
    
    return texto_normalizado

In [46]:
just_variacoes = ["justificação", "justificativa", "j u s t i f i c a ç ã o", "j u s t i f i c a ç ã", "JUSTIFICAÇÃ\nO", "JUSTIFICAÇÃO"]
plp["texto_preprocessado_sem_justificativa"] = plp['texto_preprocessado'].apply(lambda x: extract_until_any(x, just_variacoes))

In [47]:
plp[plp.texto_preprocessado == plp.texto_preprocessado_sem_justificativa]

,num_emenda,materia,nome_arquivo,texto,cabecalho,texto_sem_cabecalho,texto_preprocessado,texto_preprocessado_sem_justificativa


In [48]:
plp.to_parquet("../data/datasets/PLP_68_2024_preprocessado.parquet", index=False)

In [49]:
plp

,num_emenda,materia,nome_arquivo,texto,cabecalho,texto_sem_cabecalho,texto_preprocessado,texto_preprocessado_sem_justificativa
0,1,PLP_68_2024,EMENDA_1-U_-_PLP_68_2024.txt,Gabinete do Senador Esperidião Amin\nEMENDA Nº...,Gabinete do Senador Esperidião Amin\nEMENDA Nº...,\nSuprimam-se os §§ 1° e 2° do art. 384 do PLP...,Suprimam-se os §§ 1° e 2° do art. 384 do PLP n...,Suprimam-se os §§ 1° e 2° do art. 384 do PLP n...
1,10,PLP_68_2024,EMENDA_10-U_-_PLP_68_2024.txt,Gabinete do Senador Mecias de Jesus\nEMENDA Nº...,Gabinete do Senador Mecias de Jesus\nEMENDA Nº...,\nInclua-se o §6º no art. 23 e dê-se a seguint...,Inclua-se o §6º no art. 23 e dê-se a seguinte ...,Inclua-se o §6º no art. 23 e dê-se a seguinte ...
2,100,PLP_68_2024,EMENDA_100-U_-_PLP_68_2024.txt,Gabinete do Senador Fabiano Contarato\nEMENDA ...,Gabinete do Senador Fabiano Contarato\nEMENDA ...,\nAcrescenta o inciso VIII ao §1º do artigo 40...,Acrescenta o inciso VIII ao §1º do artigo 406 ...,Acrescenta o inciso VIII ao §1º do artigo 406 ...
3,1000,PLP_68_2024,EMENDA_1000-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nAcrescen...,EMENDA Nº \n(ao PLP 68/2024),\nAcrescente-se inciso VI ao parágrafo único d...,Acrescente-se inciso VI ao parágrafo único do ...,Acrescente-se inciso VI ao parágrafo único do ...
4,1001,PLP_68_2024,EMENDA_1001-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nDê-se ao...,EMENDA Nº \n(ao PLP 68/2024),\nDê-se ao § 7º do art. 163 do Projeto a segui...,Dê-se ao § 7º do art. 163 do Projeto a seguint...,Dê-se ao § 7º do art. 163 do Projeto a seguint...
...,...,...,...,...,...,...,...,...
2234,995,PLP_68_2024,EMENDA_995-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nDê-se ao...,EMENDA Nº \n(ao PLP 68/2024),\nDê-se ao inciso IV do caput do art. 26 do Pr...,Dê-se ao inciso IV do caput do art. 26 do Proj...,Dê-se ao inciso IV do caput do art. 26 do Proj...
2235,996,PLP_68_2024,EMENDA_996-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nAcrescen...,EMENDA Nº \n(ao PLP 68/2024),"\nAcrescente-se § 4º ao art. 450 do Projeto, c...","Acrescente-se § 4º ao art. 450 do Projeto, com...","Acrescente-se § 4º ao art. 450 do Projeto, com..."
2236,997,PLP_68_2024,EMENDA_997-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nAcrescen...,EMENDA Nº \n(ao PLP 68/2024),"\nAcrescente-se § 0º ao art. 247 do Projeto, c...","Acrescente-se § 0º ao art. 247 do Projeto, com...","Acrescente-se § 0º ao art. 247 do Projeto, com..."
2237,998,PLP_68_2024,EMENDA_998-U_-_PLP_68_2024.txt,EMENDA Nº \n(ao PLP 68/2024)\nDê-se no...,EMENDA Nº \n(ao PLP 68/2024),\nDê-se nova redação ao inciso IV do § 2º do a...,Dê-se nova redação ao inciso IV do § 2º do art...,Dê-se nova redação ao inciso IV do § 2º do art...
